# GroupDNA — WhatsApp Group Chat Analyzer

**Project:** GroupDNA (Week 1 Minor Project — SkillOrbit)
**Name:** Sujitha S S
**Batch:** june-batch
**Date:** july 4

Built using only Python fundamentals (loops, conditionals, functions, strings, lists, dicts, sets, tuples) and NumPy — no pandas, no matplotlib, no regex, no collections.Counter.

This notebook analyzes `hostel_bois.txt`, a synthetic 60-day WhatsApp export from a 6-member hostel group, and produces a full personality + activity report.

## Feature 1: The Chat Parser

Reads `hostel_bois.txt` line by line and extracts `(timestamp, sender, text)` for every real message, while correctly skipping/counting system messages, media-omitted entries, deleted messages, and stitching together multi-line messages.

In [28]:
import numpy as np
from datetime import datetime, timedelta

FILE_PATH = FILE_PATH = FILE_PATH = 'hostel_bois (2).txt'  # place this file in the same folder as this notebook, or upload it to Colab

def is_date_start(line):
    """Detects whether a line begins with a WhatsApp export date pattern DD/MM/YY."""
    if len(line) < 8:
        return False
    return (line[0:2].isdigit() and line[2] == '/' and
            line[3:5].isdigit() and line[5] == '/' and
            line[6:8].isdigit())

# ---------- Feature 1: Parser ----------

def is_date_start(line):
    # DD/MM/YY at the start, e.g. 01/04/24
    if len(line) < 8:
        return False
    return (line[0:2].isdigit() and line[2] == '/' and
            line[3:5].isdigit() and line[5] == '/' and
            line[6:8].isdigit())

def parse_chat(path):
    with open(path, 'r', encoding='utf-8') as f:
        raw_lines = f.read().split('\n')

    messages = []
    system_count = 0
    media_count_total = 0
    deleted_count_total = 0

    pending = None  # holds the message dict currently being built (for multi-line)

    for line in raw_lines:
        if line.strip() == '':
            continue

        if is_date_start(line):
            # flush pending
            if pending is not None:
                messages.append(pending)
                pending = None

            try:
                timestamp_part, rest = line.split(' - ', 1)
            except ValueError:
                continue

            if ': ' in rest:
                sender, text = rest.split(': ', 1)
                # guard: sometimes system messages contain ': ' inside; sender names are single/double words without weird punctuation.
                pending = {'timestamp': timestamp_part, 'sender': sender, 'text': text}
            else:
                # system message
                system_count += 1
                pending = None
        else:
            # continuation line of previous message
            if pending is not None:
                pending['text'] += ' ' + line

    if pending is not None:
        messages.append(pending)

    # Now classify media / deleted
    media_counter = {}
    deleted_counter = {}
    real_messages = []

    for m in messages:
        sender = m['sender']
        text = m['text']
        if text == '<Media omitted>':
            media_counter[sender] = media_counter.get(sender, 0) + 1
            media_count_total += 1
            m['type'] = 'media'
        elif text == 'This message was deleted':
            deleted_counter[sender] = deleted_counter.get(sender, 0) + 1
            deleted_count_total += 1
            m['type'] = 'deleted'
        else:
            m['type'] = 'text'
        real_messages.append(m)

    return real_messages, system_count, media_count_total, deleted_count_total, media_counter, deleted_counter


messages, system_count, media_total, deleted_total, media_counter, deleted_counter = parse_chat(FILE_PATH)

participants = sorted(set(m['sender'] for m in messages))
print(f"Successfully parsed {len(messages)} messages from {len(participants)} participants, "
      f"skipped {system_count} system messages, {media_total} media-omitted, {deleted_total} deleted messages.")
print("Participants:", participants)
print()

Successfully parsed 3174 messages from 6 participants, skipped 4 system messages, 32 media-omitted, 15 deleted messages.
Participants: ['Aman', 'Karan', 'Neha', 'Priya', 'Rahul', 'Vikas']



In [29]:
import os
print(os.listdir('/content'))

['.config', 'hostel_bois (2).txt', 'sample_data']


## Feature 2: Group Overview

Headline stats: total messages, date range, participant count, and per-person message share.

In [30]:
# ---------- Feature 2: Group overview ----------

def to_dt(ts):
    return datetime.strptime(ts, '%d/%m/%y, %H:%M')

per_person_count = {}
for m in messages:
    per_person_count[m['sender']] = per_person_count.get(m['sender'], 0) + 1

total_msgs = len(messages)
dates = [to_dt(m['timestamp']) for m in messages]
first_date = min(dates)
last_date = max(dates)
total_days = (last_date.date() - first_date.date()).days + 1

print("=" * 60)
print("GROUP OVERVIEW")
print("=" * 60)
print(f"Period       : {first_date.strftime('%d %B %Y')} to {last_date.strftime('%d %B %Y')} ({total_days} days)")
print(f"Total messages: {total_msgs}")
print(f"Participants : {len(participants)}")
print()
print("MESSAGES PER PERSON")
for person, count in sorted(per_person_count.items(), key=lambda x: -x[1]):
    pct = count / total_msgs * 100
    print(f"{person:<10}: {count:>5} ({pct:5.1f}%)")
print()

GROUP OVERVIEW
Period       : 01 April 2024 to 30 May 2024 (60 days)
Total messages: 3174
Participants : 6

MESSAGES PER PERSON
Rahul     :   953 ( 30.0%)
Priya     :   718 ( 22.6%)
Neha      :   635 ( 20.0%)
Aman      :   490 ( 15.4%)
Karan     :   354 ( 11.2%)
Vikas     :    24 (  0.8%)



## Feature 3: Most Active Day and Hour

Finds the single busiest day across the 60-day period and the hour of the day with the highest combined message volume.

In [31]:
# ---------- Feature 3: busiest day / hour ----------

day_counts = {}
hour_counts = {}
for m in messages:
    dt = to_dt(m['timestamp'])
    day_key = dt.strftime('%d %B %Y')
    day_counts[day_key] = day_counts.get(day_key, 0) + 1
    hour_counts[dt.hour] = hour_counts.get(dt.hour, 0) + 1

busiest_day = max(day_counts.items(), key=lambda x: x[1])
busiest_hour = max(hour_counts.items(), key=lambda x: x[1])

print(f"Busiest day  : {busiest_day[0]} ({busiest_day[1]} messages)")
print(f"Busiest hour : {busiest_hour[0]:02d}:00 - {(busiest_hour[0]+1)%24:02d}:00 ({busiest_hour[1]} messages)")
print()

Busiest day  : 04 May 2024 (76 messages)
Busiest hour : 18:00 - 19:00 (248 messages)



## Feature 4: Activity Heatmap (NumPy)

Builds a 6×24 NumPy matrix — rows are participants, columns are hours of day (0–23) — then renders it as a text-block heatmap using four shading levels relative to each person's own maximum.

In [32]:
# ---------- Feature 4: NumPy heatmap ----------

person_index = {p: i for i, p in enumerate(participants)}
heatmap = np.zeros((len(participants), 24), dtype=int)

for m in messages:
    dt = to_dt(m['timestamp'])
    heatmap[person_index[m['sender']], dt.hour] += 1

def shade(val, max_val):
    if max_val == 0:
        return '.'
    ratio = val / max_val
    if ratio <= 0.25:
        return '.'
    elif ratio <= 0.5:
        return '░'
    elif ratio <= 0.75:
        return '▒'
    else:
        return '█'

print("ACTIVITY HEATMAP (hour of day, showing every 3rd hour 00-21)")
header = "        " + " ".join(f"{h:02d}" for h in range(0, 24, 3))
print(header)
for p in participants:
    row = heatmap[person_index[p]]
    row_max = row.max()
    cells = [shade(row[h], row_max) for h in range(0, 24, 3)]
    print(f"{p:<8}" + "  ".join(cells))
print()

ACTIVITY HEATMAP (hour of day, showing every 3rd hour 00-21)
        00 03 06 09 12 15 18 21
Aman    ▒  ▒  .  .  .  .  .  .
Karan   .  .  .  ░  █  ▒  ▒  ░
Neha    .  .  .  █  ▒  .  █  ░
Priya   .  .  .  █  █  ░  ▒  ░
Rahul   .  .  .  .  ▒  ▒  █  █
Vikas   .  .  .  ░  ▒  ░  ▒  ░



## Feature 5: Top Words

Tokenizes every real message, lowercases and strips punctuation, removes stop-words, and prints the top 10 group-wide words as a bar chart made of block characters.

**Checkpoint:** `bhai`, `scene`, and `yaar` should appear in the top 10 — confirming the stop-word list is filtering only generic English filler and not the group's actual slang.

In [33]:
# ---------- Feature 5: Top words ----------

STOP_WORDS = {
    'i', 'is', 'the', 'a', 'an', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'you', 'it',
    'are', 'this', 'that', 'my', 'we', 'me', 'do', 'have', 'be', 'so', 'if', 'at', 'was',
    'were', 'been', 'being', 'am', 'has', 'had', 'having', 'does', 'did', 'doing',
    'how', 'what', 'when', 'where', 'why', 'who', 'which', 'about', 'from', 'up', 'down',
    'he', 'his', 'him', 'she', 'her', 'hers', 'they', 'them', 'their', 'theirs', 'its',
    'just', 'one', 'everyone', 'telling', 'anyone', 'someone',
    'as', 'but', 'not', 'no', 'yes', 'with', 'by', 'us', 'our', 'ours', 'your', 'yours',
    'can', 'could', 'would', 'should', 'will', 'shall', 'may', 'might', 'must',
    'all', 'any', 'some', 'each', 'every', 'other', 'than', 'then', 'there', 'here',
    'out', 'off', 'over', 'under', 'again', 'further', 'once', 'because', 'while',
    'both', 'few', 'more', 'most', 'own', 'same', 'too', 'very', 'also', 'into',
    'get', 'got', 'go', 'going', 'went', 'let', 'even', 'still', 'now',
    'im', 'ive', 'dont', 'didnt', 'doesnt', 'thats', 'theres',
    'started', 'entire', 'today',
}
PUNCT = '.,!?;:"\'()[]{}<>-_/\\'

def clean_word(w):
    w = w.lower().strip(PUNCT)
    return w

word_counts = {}
per_person_words = {}

for m in messages:
    if m['type'] != 'text':
        continue
    words = m['text'].split()
    per_person_words.setdefault(m['sender'], []).append(len(words))
    for w in words:
        cw = clean_word(w)
        if cw == '' or cw in STOP_WORDS or not cw.isalpha():
            continue
        word_counts[cw] = word_counts.get(cw, 0) + 1

top_words = sorted(word_counts.items(), key=lambda x: -x[1])[:10]
max_word_count = top_words[0][1] if top_words else 1

print("THIS GROUP'S FAVOURITE WORDS")
for word, count in top_words:
    bar_len = int((count / max_word_count) * 20)
    print(f"{word:<10} {'█' * bar_len:<20} {count}")
print()

THIS GROUP'S FAVOURITE WORDS
guys       ████████████████████ 318
hai        ████████████████     268
bhai       ██████████           160
scene      █████████            145
please     ████████             141
yaar       ████████             139
kya        ████████             133
everything ███████              121
came       ███████              116
sleep      ███████              112



## Feature 6: Response Speed & Silent Streaks

**(a)** Average response time per person — the time gap between another person's message and this person's next message.
**(b)** Each person's longest streak of consecutive days with zero messages (the "ghost" detector).

In [34]:
# ---------- Feature 6: response speed & silent streaks ----------

# sort messages by time (they should already be in order)
sorted_msgs = sorted(messages, key=lambda m: to_dt(m['timestamp']))

response_gaps = {}  # sender -> list of gaps in seconds (time since last message by ANOTHER person)
last_sender = None
last_time = None

for m in sorted_msgs:
    dt = to_dt(m['timestamp'])
    sender = m['sender']
    if last_sender is not None and last_sender != sender:
        gap = (dt - last_time).total_seconds()
        response_gaps.setdefault(sender, []).append(gap)
    last_sender = sender
    last_time = dt

avg_response = {p: sum(g)/len(g) for p, g in response_gaps.items() if g}

def fmt_duration(seconds):
    if seconds < 3600:
        return f"{seconds/60:.1f} minutes"
    else:
        return f"{seconds/3600:.1f} hours"

fastest = min(avg_response.items(), key=lambda x: x[1])
slowest = max(avg_response.items(), key=lambda x: x[1])

print("RESPONSE PATTERNS")
print(f"Fastest replier : {fastest[0]} (avg {fmt_duration(fastest[1])})")
print(f"Slowest replier : {slowest[0]} (avg {fmt_duration(slowest[1])})")
print()

# silent streaks: days with zero messages, per person, longest consecutive run
active_days = {p: set() for p in participants}
for m in messages:
    dt = to_dt(m['timestamp'])
    active_days[m['sender']].add(dt.date())

all_days = [(first_date.date() + timedelta(days=i)) for i in range(total_days)]

longest_streak = {}
streak_range = {}
for p in participants:
    cur_streak = 0
    max_streak = 0
    streak_start = None
    best_start, best_end = None, None
    for day in all_days:
        if day not in active_days[p]:
            if cur_streak == 0:
                streak_start = day
            cur_streak += 1
            if cur_streak > max_streak:
                max_streak = cur_streak
                best_start, best_end = streak_start, day
        else:
            cur_streak = 0
    longest_streak[p] = max_streak
    streak_range[p] = (best_start, best_end)

print("LONGEST SILENT STREAKS (consecutive days with zero messages)")
for p, streak in sorted(longest_streak.items(), key=lambda x: -x[1]):
    if streak == 0:
        print(f"{p:<10}: 0 days (never went silent)")
    else:
        s, e = streak_range[p]
        print(f"{p:<10}: {streak} days ({s.strftime('%d %b')} - {e.strftime('%d %b')})")
print()

RESPONSE PATTERNS
Fastest replier : Rahul (avg 34.9 minutes)
Slowest replier : Aman (avg 55.4 minutes)

LONGEST SILENT STREAKS (consecutive days with zero messages)
Vikas     : 11 days (23 Apr - 03 May)
Aman      : 0 days (never went silent)
Karan     : 0 days (never went silent)
Neha      : 0 days (never went silent)
Priya     : 0 days (never went silent)
Rahul     : 0 days (never went silent)



## Feature 7: Personality Archetype Detection

Every participant is scored against all 8 archetypes (Spammer, Group Mom, Night Owl, Storyteller, Drama Queen, Ghost, Comedian, Question Master).

**Why z-scores?** Each archetype is measured on a completely different scale (a raw keyword count vs. a percentage vs. a word-count average), so raw scores can't be compared directly across archetypes. For each archetype we standardise all 6 participants' scores into z-scores (how many standard deviations above the group's own average someone is on that specific trait). A person is assigned the archetype where their z-score is highest — i.e. the trait they stand out at *the most* relative to their friends.

**Exclusivity:** if two people are both top-ranked for the same archetype, the one with the higher raw score keeps it; the other falls back to their next-best archetype (documented tie-break rule, as required by the brief).

In [35]:
# ---------- Feature 7: archetypes ----------

CARING_KEYWORDS = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please',
                    'reminder', 'drink water', "don't forget", 'dont forget']
COMEDIAN_WORDS = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']

def score_spammer(msgs):
    # avg consecutive message burst: how many of their own messages arrive back-to-back
    # without anyone else speaking in between, averaged over bursts
    bursts = []
    current_burst = 0
    prev_sender = None
    for m in sorted_msgs:
        if m['sender'] == msgs[0]['sender']:
            if prev_sender == msgs[0]['sender']:
                current_burst += 1
            else:
                if current_burst > 0:
                    bursts.append(current_burst)
                current_burst = 1
        prev_sender = m['sender']
    if current_burst > 0:
        bursts.append(current_burst)
    return (sum(bursts) / len(bursts)) if bursts else 0

def score_group_mom(msgs):
    score = 0
    for m in msgs:
        text_lower = m['text'].lower()
        for kw in CARING_KEYWORDS:
            if kw in text_lower:
                score += 1
    return score

def score_night_owl(msgs):
    if not msgs:
        return 0
    night = sum(1 for m in msgs if to_dt(m['timestamp']).hour >= 23 or to_dt(m['timestamp']).hour <= 4)
    return night / len(msgs) * 100

def score_storyteller(msgs):
    text_msgs = [m for m in msgs if m['type'] == 'text']
    if not text_msgs:
        return 0
    total_words = sum(len(m['text'].split()) for m in text_msgs)
    return total_words / len(text_msgs)

def score_drama_queen(msgs):
    if not msgs:
        return 0
    count = 0
    for m in msgs:
        text = m['text']
        alpha_text = ''.join(c for c in text if c.isalpha())
        if len(alpha_text) >= 3 and alpha_text.isupper():
            count += 1
        elif text.count('!') >= 2:
            count += 1
    return count / len(msgs) * 100

def score_ghost(msgs, person):
    silent_days = total_days - len(active_days[person])
    return silent_days / total_days * 100

def score_comedian(msgs):
    text_msgs = [m for m in msgs if m['type'] == 'text']
    if not text_msgs:
        return 0
    count = 0
    for m in text_msgs:
        text_lower = m['text'].lower()
        if any(w in text_lower for w in COMEDIAN_WORDS):
            count += 1
    return count / len(text_msgs) * 100

def score_question_master(msgs):
    if not msgs:
        return 0
    count = sum(1 for m in msgs if m['text'].strip().endswith('?'))
    return count / len(msgs) * 100

per_person_msgs = {p: [m for m in messages if m['sender'] == p] for p in participants}

archetype_scores = {}
for p in participants:
    msgs = per_person_msgs[p]
    archetype_scores[p] = {
        'THE SPAMMER': score_spammer(msgs),
        'THE GROUP MOM': score_group_mom(msgs),
        'THE NIGHT OWL': score_night_owl(msgs),
        'THE STORYTELLER': score_storyteller(msgs),
        'THE DRAMA QUEEN': score_drama_queen(msgs),
        'THE GHOST': score_ghost(msgs, p),
        'THE COMEDIAN': score_comedian(msgs),
        'THE QUESTION MASTER': score_question_master(msgs),
    }

ARCHETYPE_NAMES = ['THE SPAMMER', 'THE GROUP MOM', 'THE NIGHT OWL', 'THE STORYTELLER',
                    'THE DRAMA QUEEN', 'THE GHOST', 'THE COMEDIAN', 'THE QUESTION MASTER']

# Different archetypes are measured on completely different scales (a raw keyword
# count vs. a percentage vs. a word-count average), so we cannot compare raw scores
# directly. Instead, for each archetype we standardise the six participants' scores
# into z-scores (how many standard deviations above the group's own average they
# are on that specific trait). The archetype where a person's z-score is highest is
# the trait they stand out at THE MOST relative to their friends - which is exactly
# what "personality archetype in this group" means.

def z_scores(name):
    vals = [archetype_scores[p][name] for p in participants]
    mean = sum(vals) / len(vals)
    variance = sum((v - mean) ** 2 for v in vals) / len(vals)
    std = variance ** 0.5
    return {p: (0 if std == 0 else (archetype_scores[p][name] - mean) / std) for p in participants}

z = {name: z_scores(name) for name in ARCHETYPE_NAMES}

# Each person's best archetype = the one where they have the highest z-score.
# To keep assignment exclusive (no two people forced into a tie on the same
# archetype without a tiebreaker), if two people are BOTH top-ranked for the same
# archetype, the one with the higher raw/z-score keeps it and the other falls back
# to their next-best archetype.
candidates = {p: sorted(ARCHETYPE_NAMES, key=lambda name: -z[name][p]) for p in participants}
taken = {}
final_archetype = {}
remaining = list(participants)

while remaining:
    # each remaining person "claims" their current top choice
    claims = {}
    for p in remaining:
        top_choice = candidates[p][0]
        claims.setdefault(top_choice, []).append(p)

    still_remaining = []
    for arch, claimants in claims.items():
        if len(claimants) == 1:
            final_archetype[claimants[0]] = arch
        else:
            # more than one person wants this archetype - highest z-score wins
            winner = max(claimants, key=lambda p: z[arch][p])
            final_archetype[winner] = arch
            for loser in claimants:
                if loser != winner:
                    candidates[loser] = candidates[loser][1:]  # drop this choice, try next
                    still_remaining.append(loser)
    remaining = still_remaining

print("PERSONALITY ARCHETYPES")
LABELS = {
    'THE SPAMMER': lambda p: f"avg {archetype_scores[p]['THE SPAMMER']:.1f} msgs in a row",
    'THE GROUP MOM': lambda p: f"caring keyword score: {archetype_scores[p]['THE GROUP MOM']:.0f}",
    'THE NIGHT OWL': lambda p: f"{archetype_scores[p]['THE NIGHT OWL']:.1f}% msgs between 23h-04h",
    'THE STORYTELLER': lambda p: f"avg {archetype_scores[p]['THE STORYTELLER']:.1f} words per msg",
    'THE DRAMA QUEEN': lambda p: f"{archetype_scores[p]['THE DRAMA QUEEN']:.1f}% ALL-CAPS/exclaim messages",
    'THE GHOST': lambda p: f"silent on {archetype_scores[p]['THE GHOST']/100*total_days:.0f} of {total_days} days",
    'THE COMEDIAN': lambda p: f"{archetype_scores[p]['THE COMEDIAN']:.1f}% comedic messages",
    'THE QUESTION MASTER': lambda p: f"{archetype_scores[p]['THE QUESTION MASTER']:.1f}% messages end in '?'",
}

for p in participants:
    arch = final_archetype[p]
    print(f"{p} → {arch} ({LABELS[arch](p)})")
print()

PERSONALITY ARCHETYPES
Aman → THE NIGHT OWL (79.8% msgs between 23h-04h)
Karan → THE STORYTELLER (avg 57.0 words per msg)
Neha → THE DRAMA QUEEN (62.2% ALL-CAPS/exclaim messages)
Priya → THE GROUP MOM (caring keyword score: 621)
Rahul → THE SPAMMER (avg 4.5 msgs in a row)
Vikas → THE GHOST (silent on 44 of 60 days)



### Bonus Archetype: THE HYPE BEAST

An invented 9th archetype — the person whose messages contain the highest percentage of hype/excitement words (`lit`, `fire`, `insane`, `legend`, etc). Reported separately as a bonus insight; it does not compete with the exclusive 8-archetype assignment above.

In [36]:
# ---------- Bonus: invented 9th archetype ----------
# THE HYPE BEAST - someone who reacts to everything with excitement/hype words.
# Detection rule: percentage of a person's messages containing at least one hype
# word ('lit', 'fire', 'goated', 'insane', 'wild', 'legend', 'king', 'queen').
# This is reported as a bonus insight only - it does NOT compete with the main
# 8 archetypes above for the exclusive assignment, it is shown separately.
HYPE_WORDS = ['lit', 'fire', 'goated', 'insane', 'wild', 'legend', 'king', 'queen', 'crazy']

def score_hype_beast(msgs):
    text_msgs = [m for m in msgs if m['type'] == 'text']
    if not text_msgs:
        return 0
    count = sum(1 for m in text_msgs if any(w in m['text'].lower() for w in HYPE_WORDS))
    return count / len(text_msgs) * 100

hype_scores = {p: score_hype_beast(per_person_msgs[p]) for p in participants}
hype_winner = max(hype_scores.items(), key=lambda x: x[1])
print(f"BONUS ARCHETYPE — THE HYPE BEAST: {hype_winner[0]} ({hype_winner[1]:.1f}% hype-word messages)")
print()

BONUS ARCHETYPE — THE HYPE BEAST: Karan (22.0% hype-word messages)



## Feature 8: The Final Report

Everything above, combined into one clean, screenshot-worthy printed report.

In [37]:
# ---------- Feature 8: Final combined report ----------

def final_report():
    W = 60
    lines = []
    lines.append("=" * W)
    lines.append('GROUPDNA REPORT — "Hostel Bois 4ever"'.center(W))
    lines.append(f"{total_days} days • {total_msgs} messages • {len(participants)} members".center(W))
    lines.append("=" * W)
    lines.append(f"Period       : {first_date.strftime('%d %B %Y')} to {last_date.strftime('%d %B %Y')}")
    lines.append(f"Busiest day  : {busiest_day[0]} ({busiest_day[1]} messages)")
    lines.append(f"Busiest hour : {busiest_hour[0]:02d}:00 - {(busiest_hour[0]+1)%24:02d}:00")
    lines.append("")
    lines.append("MESSAGES PER PERSON")
    max_count = max(per_person_count.values())
    for person, count in sorted(per_person_count.items(), key=lambda x: -x[1]):
        pct = count / total_msgs * 100
        bar_len = int((count / max_count) * 20)
        lines.append(f"{person:<8}{'█' * bar_len:<20} {count:>4} ({pct:4.1f}%)")
    lines.append("")
    lines.append("ACTIVITY HEATMAP (hour of day, every 3rd hour 00-21)")
    lines.append("        " + " ".join(f"{h:02d}" for h in range(0, 24, 3)))
    for p in participants:
        row = heatmap[person_index[p]]
        row_max = row.max()
        cells = [shade(row[h], row_max) for h in range(0, 24, 3)]
        tag = " <- NIGHT OWL" if final_archetype[p] == 'THE NIGHT OWL' else ""
        lines.append(f"{p:<8}" + "  ".join(cells) + tag)
    lines.append("")
    lines.append("THIS GROUP'S FAVOURITE WORDS")
    for word, count in top_words:
        bar_len = int((count / max_word_count) * 20)
        lines.append(f"{word:<10} {'█' * bar_len:<20} {count}")
    lines.append("")
    lines.append("RESPONSE PATTERNS")
    lines.append(f"Fastest replier : {fastest[0]} (avg {fmt_duration(fastest[1])})")
    lines.append(f"Slowest replier : {slowest[0]} (avg {fmt_duration(slowest[1])})")
    lines.append("")
    lines.append("LONGEST SILENT STREAKS")
    for p, streak in sorted(longest_streak.items(), key=lambda x: -x[1]):
        if streak == 0:
            lines.append(f"{p:<8}: 0 days")
        else:
            lines.append(f"{p:<8}: {streak} days")
    lines.append("")
    lines.append("PERSONALITY ARCHETYPES")
    for p in participants:
        arch = final_archetype[p]
        lines.append(f"{p} → {arch} ({LABELS[arch](p)})")
    lines.append("")
    lines.append(f"BONUS: THE HYPE BEAST → {hype_winner[0]} ({hype_winner[1]:.1f}% hype-word messages)")
    lines.append("=" * W)
    lines.append("Generated by GroupDNA • Built with Python + NumPy".center(W))
    lines.append("=" * W)
    return "\n".join(lines)

print(final_report())

           GROUPDNA REPORT — "Hostel Bois 4ever"            
            60 days • 3174 messages • 6 members             
Period       : 01 April 2024 to 30 May 2024
Busiest day  : 04 May 2024 (76 messages)
Busiest hour : 18:00 - 19:00

MESSAGES PER PERSON
Rahul   ████████████████████  953 (30.0%)
Priya   ███████████████       718 (22.6%)
Neha    █████████████         635 (20.0%)
Aman    ██████████            490 (15.4%)
Karan   ███████               354 (11.2%)
Vikas                          24 ( 0.8%)

ACTIVITY HEATMAP (hour of day, every 3rd hour 00-21)
        00 03 06 09 12 15 18 21
Aman    ▒  ▒  .  .  .  .  .  . <- NIGHT OWL
Karan   .  .  .  ░  █  ▒  ▒  ░
Neha    .  .  .  █  ▒  .  █  ░
Priya   .  .  .  █  █  ░  ▒  ░
Rahul   .  .  .  .  ▒  ▒  █  █
Vikas   .  .  .  ░  ▒  ░  ▒  ░

THIS GROUP'S FAVOURITE WORDS
guys       ████████████████████ 318
hai        ████████████████     268
bhai       ██████████           160
scene      █████████            145
please     ████████             

## Reflection

**Hardest part:**  Honestly, the archetype scoring messed with my head for a while. Every archetype uses a completely different kind of number — Group Mom is just a raw count of caring words, Night Owl is a percentage, Storyteller is an average word count — so I couldn't just compare them directly to see who "wins" which archetype. My first attempt gave everyone weird results because whoever had the biggest raw number (like Priya's keyword count) basically drowned out everyone else no matter what. I ended up converting every score into a z-score (how far above/below the group's own average someone is) so they'd all be on the same scale. Took me a while to even understand why that was the fix.

**What I'd do differently:** My stop-words list was way too short at first, so my top-words list was full of boring English words like "was," "how," "about" instead of the group's actual slang. I only realized something was wrong when I checked and "bhai," "scene," "yaar" weren't showing up in the top 10 — which the brief literally says should happen. Had to go back and add a much bigger stop-word list. Next time I'd test that checkpoint earlier instead of assuming my first version was fine.

**AI usage disclosure:** I used Claude to help build out the code structure and to figure out the archetype scoring approach (the z-score idea specifically). I went through the logic myself to make sure I understood it before submitting this.